# Chapter 5 &mdash; The Mod Algebra That Makes Residue States Work

**Concept 8 of the Chapter 5 decomposition:** *The Mod Algebra That Makes Residue States Computable*

$(a+b)\%N$ and $(ab)\%N$ can be computed from residues alone &mdash; which is exactly why finite memory suffices.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Mod-Algebra/Concept-Mod-Algebra.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Residue states are only legitimate because of two identities:

$$(a+b)\bmod N = ((a\bmod N) + (b\bmod N))\bmod N$$
$$(a\cdot b)\bmod N = ((a\bmod N)\cdot(b\bmod N))\bmod N$$

They say the residue of a result depends **only on the residues of the inputs**, never
on the full values. That is a **homomorphism** from $\mathbb{Z}$ onto
$\mathbb{Z}_N$ &mdash; and it is precisely the property that lets a *finite* machine track
an *infinite* quantity.

When a condition has no such algebra (like "equal numbers of 0s and 1s"), no residue
trick exists, and the language turns out non-regular.

## 2. Definitions

### The two identities, as checkable predicates

In [ ]:
def add_ok(a, b, N): return (a + b) % N == ((a % N) + (b % N)) % N
def mul_ok(a, b, N): return (a * b) % N == ((a % N) * (b % N)) % N

### Why the DFA recurrence is a consequence

In [ ]:
def resid_from_scratch(s, m): return int(s, 2) % m if s else 0
def resid_incremental(s, m):
    r = 0
    for b in s: r = (2*r + int(b)) % m      # uses BOTH identities
    return r

## 3. Tests

The identities hold, exhaustively on a decent range.

In [ ]:
import random
assert all(add_ok(a, b, N) and mul_ok(a, b, N)
           for N in range(2, 12) for a in range(60) for b in range(60))
print("both identities verified for N in 2..11, a,b in 0..59")
for a, b, N in [(17, 25, 3), (1000, 999, 7)]:
    print("  (%d+%d)%%%d = %d = (%d+%d)%%%d" % (a,b,N,(a+b)%N,a%N,b%N,N))

Therefore the incremental residue equals the true residue &mdash; the DFA is justified.

In [ ]:
from itertools import product
for m in [3, 5, 7]:
    bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
           if resid_incremental(''.join(p), m) != resid_from_scratch(''.join(p), m)]
    print("m=%d  mismatches: %d" % (m, len(bad)))
    assert not bad
print("\nThe state never holds N -- only N mod m -- and that is provably enough.")

Where the algebra runs out: a difference of counts has no bounded residue.

In [ ]:
def diff(s): return s.count('0') - s.count('1')
vals = {diff('0'*k + '1'*j) for k in range(12) for j in range(12)}
print("values the 'difference' quantity can take :", sorted(vals)[:8], "...")
print("unbounded -> no finite residue set -> no DFA.  (Chapter 4's L_01.)")

## 4. Exercises


1. Does the subtraction identity hold too? Check it for negative $a-b$.
2. Which algebraic property fails for "number of 0s minus number of 1s"?
3. State the homomorphism $\mathbb{Z}\to\mathbb{Z}_N$ precisely.

In [ ]:
# Your work for the exercises above.